# ARCHS4 model building with CLAMP (CLAMP_K × 4)

💡 **Environment:** `clamp-analyses`  

This notebook runs CLAMPbase on ARCHS4 data using `CLAMP_K * 4`, where `CLAMP_K` is the value estimated in notebook `03_archs4_CLAMPbase.ipynb` (i.e., `num.pc() * 2`). Here we multiply that value by 4 to explore the effect of a larger latent space.

## Load libraries

In [ ]:
if (!requireNamespace("CLAMP", quietly = TRUE)) {
    devtools::install_github("wgmao/CLAMP")
}

library(bigstatsr)
library(data.table)
library(dplyr)
library(rsvd)
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(CLAMP)

source(here("config.R"))

set.seed(config$ARCHS4$RANDOM_SVD_SEED)

## Output directory

In [ ]:
output_dir <- config$ARCHS4$DATASET_FOLDER
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

## Input data

In [ ]:
# meta
meta <- readRDS(file.path(output_dir, "metadata_filtered.rds"))
n_genes_thin <- meta$n_genes_thin
n_samples <- meta$n_samples

# fbm
fbm_file  <- file.path(output_dir, "fbm")
output_file <- paste0(fbm_file, "_filtered")

archs4_fbm_filt <- FBM(
  nrow        = n_genes_thin,
  ncol        = n_samples,
  backingfile = output_file,
  create_bk   = FALSE,
)

# svd
archs4_svdRes <- readRDS(file.path(output_dir, "svd.rds"))

In [ ]:
all_samples <- readRDS(file.path(output_dir, "all_samples.rds"))

## Set CLAMP_K × 4

In [ ]:
CLAMP_K_archs4 <- num.pc(list(d = archs4_svdRes$d)) * 8
message("Inferred CLAMP K = ", CLAMP_K_archs4)

In [ ]:
saveRDS(CLAMP_K_archs4, file = file.path(output_dir, "CLAMP_K_archs4_K4x.rds"))

## CLAMPbase initialization

In [ ]:
archs4_baseRes <- CLAMPbase(
  Y = archs4_fbm_filt,
  svdres = archs4_svdRes,
  trace  = TRUE,
  clamp_k = CLAMP_K_archs4
)

In [ ]:
archs4_genes <- meta$gene_symbols_thin
sample_names <- all_samples[seq_len(n_samples)]

In [ ]:
archs4_baseRes$Z <- data.frame(archs4_baseRes$Z)
rownames(archs4_baseRes$Z) <- archs4_genes
head(archs4_baseRes$Z)

archs4_baseRes$B <- data.frame(archs4_baseRes$B)
colnames(archs4_baseRes$B) <- sample_names
head(archs4_baseRes$B)

In [ ]:
saveRDS(archs4_baseRes, file = file.path(output_dir, "archs4_baseRes_K4x.rds"))

In [ ]:
model_dir <- file.path(output_dir, "CLAMPbase_K4x")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)

B <- archs4_baseRes$B
write.csv(B, file.path(model_dir, "B.csv"))

Z <- archs4_baseRes$Z
write.csv(Z, file.path(model_dir, "Z.csv"))